# 3.2 · Resources and resource templates

**资源（Resources）与资源模板**

> **After this lesson you can:** Expose data as URI-addressed resources and RFC 6570 templates, serve text and binary contents with the right metadata, and handle not-found, annotations and change notifications correctly.

Official reading: [Spec: Resources](https://modelcontextprotocol.io/specification/2026-07-28/server/resources) · [Understanding MCP servers](https://modelcontextprotocol.io/docs/2026-07-28/learn/server-concepts)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. A server with direct resources and templates
`MCPServer` decides by the URI: no `{...}` means a direct resource (in `resources/list`), any variable makes it a template (in `resources/templates/list`). Returning `str` gives `text`, returning `bytes` gives a base64 `blob`. We also register a real file with `FileResource` under a `file://` URI.

In [ ]:
import base64
import json
import logging
import struct
import tempfile
import zlib
from pathlib import Path

import mcp_types as types
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver.resources.types import FileResource
from mcp.shared.exceptions import MCPError

logging.getLogger("mcp").setLevel(logging.CRITICAL)


def show(model, limit=1500):
    data = model.model_dump(by_alias=True, exclude_none=True, mode="json")
    data.pop("_meta", None)
    print(json.dumps(data, indent=2, ensure_ascii=False)[:limit])


def tiny_png() -> bytes:
    chunk = lambda kind, body: (struct.pack(">I", len(body)) + kind + body
                                + struct.pack(">I", zlib.crc32(kind + body) & 0xFFFFFFFF))
    return (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", 1, 1, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(b"\x00\xff\x00\x00")) + chunk(b"IEND", b""))


project = Path(tempfile.mkdtemp())
(project / "README.md").write_text("# Trip planner\nPlan trips with MCP.\n")

server = MCPServer("travel-data")
server.add_resource(FileResource(uri=(project / "README.md").as_uri(), path=project / "README.md",
                                 name="README.md", title="Project README", mime_type="text/markdown"))


@server.resource("calendar://events/2026", title="2026 calendar", mime_type="application/json",
                 annotations=types.Annotations(audience=["assistant"], priority=0.9,
                                               last_modified="2026-07-28T09:00:00Z"))
def calendar() -> str:
    return json.dumps({"busy": ["2026-08-03", "2026-08-04"]})


@server.resource("travel://preferences", title="Travel preferences",
                 annotations=types.Annotations(audience=["user", "assistant"], priority=0.6))
def preferences() -> str:
    return "Window seat. Prefers boutique hotels. Vegetarian."


@server.resource("images://logo.png", title="Company logo", mime_type="image/png",
                 annotations=types.Annotations(audience=["user"], priority=0.1))
def logo() -> bytes:
    return tiny_png()


@server.resource("weather://forecast/{city}/{date}", title="Weather Forecast", mime_type="application/json",
                 description="Get weather forecast for any city and date")
def forecast(city: str, date: str) -> str:
    return json.dumps({"city": city, "date": date, "high_c": 29, "conditions": "sunny"})


@server.resource("travel://activities/{city}/{category}", title="Activities", mime_type="application/json")
def activities(city: str, category: str) -> str:
    catalogue = {"museums": ["Picasso Museum", "MNAC"], "beaches": ["Barceloneta", "Bogatell"]}
    return json.dumps({"city": city, "category": category, "items": catalogue.get(category, [])})


DOCS = {"guides/packing.md": "Bring an adapter.", "guides/visas.md": "Check visa rules early."}


@server.resource("docs://{+path}", title="Travel guides", mime_type="text/markdown")
def docs(path: str) -> str:
    return DOCS[path]


print("defined:", server.name)

### 2. Discovery: direct resources vs templates
Two different list calls. Notice the annotations, and that templates carry `uriTemplate` instead of `uri`.

In [ ]:
async with Client(server) as client:
    print("capabilities.resources:", client.server_capabilities.resources)
    direct = await client.list_resources()
    print("\nresources/list:")
    for r in direct.resources:
        a = r.annotations.model_dump(by_alias=True, exclude_none=True) if r.annotations else {}
        print(f"  {r.uri:<44} {r.mime_type:<18} {a}")
    templates = await client.list_resource_templates()
    print("\nresources/templates/list:")
    for t in templates.resource_templates:
        print(f"  {t.uri_template:<44} {t.mime_type}")

### 3. Read text, binary and templated resources
Templates are expanded **client-side** with RFC 6570 rules (here with the SDK's `UriTemplate`; note the percent-encoding of the space and `ã`), then read with ordinary `resources/read`.

In [ ]:
from mcp.shared.uri_template import UriTemplate

async with Client(server) as client:
    show(await client.read_resource("calendar://events/2026"))

    blob = (await client.read_resource("images://logo.png")).contents[0]
    raw = base64.b64decode(blob.blob)
    png_ok = raw.startswith(bytes([0x89]) + b"PNG")
    print(f"\nblob: mimeType={blob.mime_type}, {len(raw)} bytes after base64 decoding, PNG signature ok: {png_ok}")

    uri = UriTemplate.parse("weather://forecast/{city}/{date}").expand({"city": "São Paulo", "date": "2026-08-01"})
    print("\nexpanded:", uri)
    print("contents:", (await client.read_resource(uri)).contents[0].text)

    uri = UriTemplate.parse("docs://{+path}").expand({"path": "guides/packing.md"})
    print("reserved expansion keeps slashes:", uri, "->", (await client.read_resource(uri)).contents[0].text)

### 4. Not found is `-32602` — and never an empty array
Read a URI that matches nothing and one where the template matches but the data is missing. Then see the same error as raw JSON on the wire, via a stdio subprocess.

In [ ]:
async with Client(server) as client:
    for uri in ["calendar://events/1999", "docs://guides/nonexistent.md"]:
        try:
            await client.read_resource(uri)
        except MCPError as err:
            print(f"{uri:<32} -> code={err.code} message={err.message!r} data={err.data}")

The second case shows an SDK detail: a template handler that raises (here a `KeyError`) becomes `-32603` *internal error*, not not-found. To follow the spec, raise the SDK's `ResourceNotFoundError` from the handler:

In [ ]:
from mcp.server.mcpserver.exceptions import ResourceNotFoundError


@server.resource("guides://{+path}", mime_type="text/markdown")
def guides(path: str) -> str:
    if path not in DOCS:
        raise ResourceNotFoundError(f"Resource not found: guides://{path}")
    return DOCS[path]


async with Client(server) as client:
    try:
        await client.read_resource("guides://nope.md")
    except MCPError as err:
        print("fixed:", err.code, err.message, err.data)


def is_not_found(err: MCPError) -> bool:
    """Client-side: accept the 2026-07-28 code and the legacy one."""
    return err.code in (-32602, -32002)


print("treat as not found:", is_not_found(MCPError(-32002, "Resource not found")))

In [ ]:
import subprocess
import sys
import textwrap

workdir = Path(tempfile.mkdtemp())
(workdir / "res_server.py").write_text(textwrap.dedent('''
    from mcp.server import MCPServer
    server = MCPServer("res")

    @server.resource("config://app", mime_type="application/json")
    def config() -> str:
        return '{"theme": "dark"}'

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))
proc = subprocess.Popen([sys.executable, str(workdir / "res_server.py")], stdin=subprocess.PIPE,
                        stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, text=True)
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}
for i, uri in enumerate(["config://app", "config://missing"], 1):
    msg = {"jsonrpc": "2.0", "id": i, "method": "resources/read", "params": {"uri": uri, "_meta": META}}
    proc.stdin.write(json.dumps(msg) + "\n")
    proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    reply.get("result", {}).pop("_meta", None)
    print(">>>", json.dumps({k: v for k, v in msg["params"].items() if k != "_meta"}))
    print("<<<", json.dumps(reply), "\n")
proc.stdin.close()
_ = proc.wait(timeout=10)

### 5. Break it: directory traversal
The spec says servers MUST sanitise paths. The SDK's templates reject `..` components and absolute paths in extracted parameters by default (`ResourceSecurity`), so the request never reaches our handler.

In [ ]:
async with Client(server) as client:
    for uri in ["docs://guides/visas.md", "docs://../../etc/passwd", "docs:///etc/passwd", "docs://guides/%2e%2e/%2e%2e/secret"]:
        try:
            text = (await client.read_resource(uri)).contents[0].text
            print(f"{uri:<38} OK   {text}")
        except MCPError as err:
            print(f"{uri:<38} {err.code} {err.message}")

If you build paths yourself (a custom low-level handler, a `FileResource` factory), do the same check: resolve the path and verify it is still inside the root.

In [ ]:
def safe_join(root: Path, relative: str) -> Path:
    candidate = (root / relative).resolve()
    if not candidate.is_relative_to(root.resolve()):
        raise PermissionError(f"path escapes root: {relative}")
    return candidate


for rel in ["README.md", "../../etc/passwd"]:
    try:
        print(rel, "->", safe_join(project, rel))
    except PermissionError as exc:
        print(rel, "->", exc)

### 6. A host that picks context with annotations
Resources are application-driven: the host chooses. This tiny selector keeps only items meant for the assistant, orders by priority, and stops at a character budget.

In [ ]:
async def select_context(client, budget_chars=120):
    resources = (await client.list_resources()).resources
    for_model = [r for r in resources
                 if not r.annotations or not r.annotations.audience or "assistant" in r.annotations.audience]
    ranked = sorted(for_model, key=lambda r: (r.annotations.priority if r.annotations and r.annotations.priority is not None else 0.5),
                    reverse=True)
    chosen, used = [], 0
    for r in ranked:
        item = (await client.read_resource(str(r.uri))).contents[0]
        text = getattr(item, "text", None)
        if text is None or used + len(text) > budget_chars:
            print(f"skip   {r.uri} ({'binary' if text is None else 'over budget'})")
            continue
        chosen.append((str(r.uri), text))
        used += len(text)
        print(f"attach {r.uri} ({len(text)} chars)")
    return chosen


async with Client(server) as client:
    context = await select_context(client)
print("\nprompt context:\n" + "\n".join(f"<resource uri='{u}'>{t.strip()}</resource>" for u, t in context))

The logo (`audience: ["user"]`) is never considered for the model, the calendar (priority 0.9) goes first.

### 7. Preview: watching a resource for changes
In 2026-07-28 a client watches URIs with `subscriptions/listen` (lesson 3.4 has the details). A tool changes the calendar and publishes `notifications/resources/updated`; the notification names only the URI, so the client re-reads.

In [ ]:
from mcp.server.mcpserver import Context

busy_days = ["2026-08-03", "2026-08-04"]


@server.resource("calendar://busy", mime_type="application/json")
def busy() -> str:
    return json.dumps(busy_days)


@server.tool()
async def block_day(day: str, ctx: Context) -> str:
    """Mark a day as busy in the calendar."""
    busy_days.append(day)
    await ctx.notify_resource_updated("calendar://busy")
    return f"blocked {day}"


async with Client(server) as client:
    print("before:", (await client.read_resource("calendar://busy")).contents[0].text)
    async with client.listen(resource_subscriptions=["calendar://busy"]) as sub:
        await client.call_tool("block_day", {"day": "2026-08-10"})
        event = await anext(sub)
        print("event:", event)
        print("after: ", (await client.read_resource(event.uri)).contents[0].text)

### Your turn
1. Add a template `travel://flights/{origin}/{destination}` and a direct resource with `audience: ["user", "assistant"]` and `priority: 1.0`. Re-run sections 2 and 6: where does the new resource land in the selection?
2. Add a template using a query expansion, `search://hotels{?city,max_price}` (the handler parameters need defaults). Expand it with `UriTemplate` for two different inputs and read both.
3. Make `forecast` reject dates that are not `YYYY-MM-DD` with `ResourceNotFoundError`, and decide: is "bad format" really *not found*, or would a different error code be more honest?
4. In section 4's raw stdio cell, send a `resources/read` without the `uri` parameter. What code comes back, and how does it differ from a missing resource?